# Thai GEC Evaluation (Refactored)
This notebook refactors the evaluation of Thai GEC models. It supports dynamic paths and evaluates using both original (Ver 1) and refined (Ver 2) custom token-level F0.5 metrics, alongside BLEU, GLEU, and CHRF.

## 1. Setup & Installation

In [ ]:
!pip install nltk pythainlp pandas

In [ ]:
import csv
import re
from collections import Counter
from pythainlp.tokenize import word_tokenize
import difflib
import pandas as pd
import nltk
from nltk.translate.bleu_score import corpus_bleu
from nltk.translate.gleu_score import corpus_gleu
from nltk.translate.chrf_score import corpus_chrf

## 2. Configuration & Paths
Set the path to your CSV data folder. For testing, we use the local `csv_data` directory.

In [ ]:
# You can change these paths as needed.
BASE_DIR = '../csv_data'
REFERENCE_CSV = f'{BASE_DIR}/ctfl_cleaned_test.csv'

# Dictionary mapping Model Name -> Path to its predicted CSV
MODELS_TO_EVALUATE = {
    'Gemini-1.0-Pro-ZS': f'{BASE_DIR}/ctfl_cleaned_test - gemini.csv',
    'MT5-base-FT': f'{BASE_DIR}/mt5-base-finetuned-ctfloutput.csv',
    'MT5-large-SI-100%': f'{BASE_DIR}/mt5-large-finetuned-ctfl-augmented_1output.csv'
    # Add other models here...
}

## 3. Data Loading & Preprocessing

In [ ]:
def load_reference(csv_file):
    original, target = [], []
    with open(csv_file, "r", newline="", encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader) # Skip header
        for row in reader:
            if len(row) > 1:
                original.append(word_tokenize(row[2]))
                target.append(word_tokenize(row[3]))
    return original, target

def load_prediction(csv_file):
    preds = []
    with open(csv_file, "r", newline="", encoding="utf-8") as file:
        reader = csv.reader(file)
        rows = list(reader)
        # Check for header
        if rows and len(rows[0]) > 0 and rows[0][0].lower() in ['text', 'writing_id', 'source', 'text']:
            rows = rows[1:]
            
        for row in rows:
            if len(row) > 1:
                text = row[1]
                
                # Model specific cleanups (Typhoon, etc.)
                splits = text.split('\n')
                if len(splits) > 1:
                    text = re.sub("แก้ไขประโยคต่อไปนี้ให้ถูกต้องตามไวยากรณ์ แสดงผลเฉพาะประโยคที่แก้ไขแล้ว","",splits[1])
                else:
                    text = re.sub("แก้ไขประโยคต่อไปนี้ให้ถูกต้องตามไวยากรณ์ แสดงผลเฉพาะประโยคที่แก้ไขแล้ว","",text)
                
                if "#" in text:
                    text = text.split("#")[0]
                
                # Pad/Token cleaning
                text = text.strip('<pad> ').strip('edit:').strip('แก้ไข:').strip('</s>').strip('|').strip(' ')
                preds.append(word_tokenize(text))
    return preds

## 4. Evaluation Metrics (Ver 1 & Ver 2)

In [ ]:
# --- VER 1 (Original Custom Metric) ---
def get_dif_v1(original, target):
    dif = []
    for i in range(len(original)):
        a = original[i]
        b = target[i]
        count_a = Counter(a)
        count_b = Counter(b)
        output = []
        for word in count_b:
            if count_a[word] != count_b[word]:
                for j in range(abs(count_b[word]-count_a[word])):
                    output.append(word)
        dif.append(output)
    return dif

def calculate_v1(ty2, t2):
    tp, fp, fn = 0, 0, 0
    for i in range(len(ty2)):
        if not ty2[i]: continue
        for m2_edit in ty2[i]:
            if m2_edit in t2[i]:
                tp += 1
            else:
                fp += 1

    for i in range(len(t2)):
        if not t2[i]: continue
        for m2_edit in t2[i]:
            if m2_edit not in ty2[i]:
                fn += 1
                
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall = tp / (tp + fn) if tp + fn > 0 else 0
    f1 = ((1 + (1**2)) * precision * recall) / (((1**2) * precision) + recall) if precision + recall else 0.0
    f05 = ((1 + (0.5**2)) * precision * recall) / (((0.5**2) * precision) + recall) if precision + recall else 0.0
    return precision, recall, f1, f05

# --- VER 2 (Refined with difflib) ---
def get_dif_v2(original, target):
    edits = []
    for i in range(len(original)):
        a = original[i]
        b = target[i]
        sm = difflib.SequenceMatcher(None, a, b)
        sentence_edits = []
        for tag, i1, i2, j1, j2 in sm.get_opcodes():
            if tag == 'equal': continue
            deleted = tuple(a[i1:i2])
            inserted = tuple(b[j1:j2])
            sentence_edits.append((tag, deleted, inserted))
        edits.append(sentence_edits)
    return edits

def calculate_v2(ty2, t2):
    tp, fp, fn = 0, 0, 0
    for i in range(len(ty2)):
        pred_edits = ty2[i]
        ref_edits = t2[i]
        
        pred_counter = Counter(pred_edits)
        ref_counter = Counter(ref_edits)
        
        overlap = pred_counter & ref_counter
        tp += sum(overlap.values())
        fp += sum((pred_counter - ref_counter).values())
        fn += sum((ref_counter - pred_counter).values())

    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall = tp / (tp + fn) if tp + fn > 0 else 0
    f1 = ((1 + (1**2)) * precision * recall) / (((1**2) * precision) + recall) if precision + recall else 0.0
    f05 = ((1 + (0.5**2)) * precision * recall) / (((0.5**2) * precision) + recall) if precision + recall else 0.0
    return precision, recall, f1, f05

## 5. Evaluate All Models & Generate Report

In [ ]:
results = []
orig, target = load_reference(REFERENCE_CSV)
target_refs_for_bleu = [[x] for x in target]

# Precompute target differences
target_dif_v1 = get_dif_v1(orig, target)
target_dif_v2 = get_dif_v2(orig, target)

for model_name, csv_path in MODELS_TO_EVALUATE.items():
    try:
        preds = load_prediction(csv_path)
        
        # Make sure preds match original length
        if len(preds) > len(orig):
            preds = preds[:len(orig)]
        elif len(preds) < len(orig):
            print(f"Warning: {model_name} has {len(preds)} rows, expected {len(orig)}. Truncating...")
            orig_trunc, target_trunc = orig[:len(preds)], target[:len(preds)]
            t_dif_v1 = get_dif_v1(orig_trunc, target_trunc)
            t_dif_v2 = get_dif_v2(orig_trunc, target_trunc)
            refs_bleu = target_refs_for_bleu[:len(preds)]
        else:
            t_dif_v1 = target_dif_v1
            t_dif_v2 = target_dif_v2
            refs_bleu = target_refs_for_bleu
            
        pred_dif_v1 = get_dif_v1(orig, preds)
        p1, r1, f1_1, f05_1 = calculate_v1(pred_dif_v1, t_dif_v1)
        
        pred_dif_v2 = get_dif_v2(orig, preds)
        p2, r2, f1_2, f05_2 = calculate_v2(pred_dif_v2, t_dif_v2)
        
        # Calculate standard MT metrics
        bleu = corpus_bleu(refs_bleu, preds)
        gleu = corpus_gleu(refs_bleu, preds)
        # Note: nltk corpus_chrf expects single strings or lists of lists of strings, we'll join tokens
        refs_chrf = [[" ".join(t)] for t in target]
        preds_chrf = [" ".join(p) for p in preds]
        chrf = corpus_chrf(refs_chrf, preds_chrf)
        
        results.append({
            'Model': model_name,
            'P (v1)': round(p1, 2),
            'R (v1)': round(r1, 2),
            'F1 (v1)': round(f1_1, 2),
            'F0.5 (v1)': round(f05_1, 2),
            'P (v2)': round(p2, 2),
            'R (v2)': round(r2, 2),
            'F1 (v2)': round(f1_2, 2),
            'F0.5 (v2)': round(f05_2, 2),
            'BLEU': round(bleu, 2),
            'GLEU': round(gleu, 2),
            'CHRF': round(chrf, 2)
        })
    except Exception as e:
        print(f"Error evaluating {model_name}: {e}")

df = pd.DataFrame(results)
display(df)

## 6. Export to LaTeX

In [ ]:
# Export the V2 metrics table to LaTeX
latex_df = df[['Model', 'P (v2)', 'R (v2)', 'F1 (v2)', 'F0.5 (v2)', 'BLEU', 'GLEU', 'CHRF']]
latex_df.columns = ['Model', 'P', 'R', 'F1', 'F0.5', 'BLEU', 'GLEU', 'CHRF']

latex_table = latex_df.to_latex(index=False, float_format="%.2f")
print(latex_table)
